# Paths and Text Files

You will describe file locations and write, append, and verify saved text using a consistent encoding.

CSC-239 · Module 8 · Lesson 1 of 3

A file gives text a life beyond one program run. You will begin by separating a location from its contents. Then you will save a roster, add one person, and read back exactly what was stored. Guided tasks lead to your own saved menu and tests of replacement, empty text, and repeated runs.

This lesson uses your earlier work with Strings, static methods, and checked exception handlers. The examples create their own practice directories. Those directories remain after a cell ends; cleanup is taught later in this module.

[Review the module’s text-file vocabulary](terms.md).


## Learning Goals

- Create and resolve paths inside a new practice directory.
- Write, append, and read back UTF-8 text while handling I/O failure.


## Why This Matters

A program that keeps a roster only in a variable loses access to that value when its run ends. Saving text lets another run recover the record. Choosing a location and a shared encoding makes that saved information usable by more than the code that first wrote it.

The choice between replacement and append changes what an application preserves. Replacing an attendance list can correct it completely; appending can record a new arrival. Confusing those operations can lose earlier entries or add duplicates. Reading the saved text back gives evidence about the result, including line breaks that are easy to miss by eye.

These operations prepare you to load a word dictionary for the Ghost project. The next lesson processes one line at a time; first, you need dependable file setup and a clear rule for how text is stored.


## Check Your Starting Point

Use your earlier String and exception knowledge. For `String names = "Maya\nBo\n";`, explain how many positions `names.length()` counts and why `System.out.print(names)` shows two lines. Then explain what `names.equals("Maya\nBo\n")` checks. Finally, if a statement inside a `try` block throws an exception handled by its `catch`, do the remaining statements in that `try` run before the handler? Explain.


In [ ]:
Your response:

String length and newline reasoning:


What equals checks:


Control flow after the exception:


<details>
<summary>Show answer</summary>

The count is 8: four letters and a newline for Maya, then two letters and a newline for Bo. `print` uses the two newline characters already in the String. `equals` checks the complete text, including those newlines. It does not merely compare the names as visible words.

An exception leaves the unfinished part of the `try`. Its matching handler runs; the skipped statements do not run first. The file examples use this same rule when an operation fails. A common mistake is treating a handled exception as if it supplied the missing result and resumed the failed statement.

</details>


## Video Demonstration

Follow the path from a new practice directory to a saved roster. The demonstration explains the first write, append, and read-back before a separate prediction case.

<video controls preload="metadata" width="960">
<source src="media/01_paths_and_text_files/demo.mp4" type="video/mp4">
<track kind="captions" src="media/01_paths_and_text_files/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the paths and text files demonstration transcript](media/01_paths_and_text_files/transcript.md).


## Concept

### Separate a location from its contents

A club coordinator starts a roster with Maya and Luis, then adds Nora. Each name must occupy its own newline-terminated line. The coordinator needs the saved text, not just a message saying that a write was attempted. We will work toward that roster after establishing how Java identifies its location.

A **file system** organizes stored files and directories. A **directory** groups entries such as files and other directories. A **file path** describes a location in that system. Java’s **`Path`** type represents a path; its value is separate from any file contents at that location.

```java
Path relative = Path.of("roster.txt");
```

This illustrative line needs `import java.nio.file.Path;`, as shown in the complete cell below. The keyword **`import`** lets us use a library type’s short name. `Path.of` is a static library method call: supply the name, receive a Path value, and store it in `relative`. No file is created by that assignment.

A **relative path** needs a base location for interpretation. Here, `roster.txt` is interpreted from the program’s current working directory, the location used as its starting point. An **absolute path** includes a starting location; on this system it starts from the root `/`. Neither form proves that a file exists.

The complete example asks for the final name with `getFileName()` and tests the path form with `isAbsolute()`. Both inspect the Path value. They do not read or write file contents.


In [ ]:
import java.nio.file.Path;
Path relative = Path.of("roster.txt");
System.out.println(relative.getFileName());
System.out.println("Absolute: " + relative.isAbsolute());


Expected output:

```text
roster.txt
Absolute: false
```

The final name is predictable. The `false` result says that this particular path is relative; it says nothing about whether a file named roster.txt exists.


### Create a separate place for one example

Using a file already on your computer would mix the example with unrelated information. A **temporary practice directory** gives one complete run its own location. A **fixture** is controlled starting data or state for a demonstration or test.

These illustrative lines belong inside the complete handler shown below:

```java
Path directory = Files.createTempDirectory("csc239-start-");
Path file = directory.resolve("notes.txt");
```

`Files` supplies static methods that perform file operations. `createTempDirectory` actually creates a new directory and returns its Path. The supplied prefix helps identify the purpose; generated characters make each new directory distinct. We keep the returned location in `directory` so later operations use that exact place.

**Path resolution** combines a base path with a child path. In the second line, `resolve` combines the new directory with the fixed relative child name `notes.txt`. It returns a Path stored in `file`. The directory exists now, but resolving the child does not create notes.txt. This example uses a chosen relative name; resolution alone is not a general check for an untrusted name supplied by someone else.

The full program needs the failure pattern you learned in Module 7. **I/O** means input/output; here it refers to communication with stored files. **`IOException`** is a checked exception type used to report an I/O failure. The keywords **`try`** and **`catch`** keep the attempted work and its matching handler together. If directory creation fails, Java skips the resolution and normal print, then the handler reports `File problem: ` plus the exception’s message.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-start-");
    Path file = directory.resolve("notes.txt");
    System.out.println(file.getFileName());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
notes.txt
```

The print inspects the last part of the child path. It avoids an exact full-directory expectation because the generated part changes. Only the directory was created. The handler does not run on this successful path.

“Temporary” describes the intended use and location. Java does not automatically remove this directory when the cell finishes. The module’s final lesson introduces removal of the files and directories created by an exercise. For now, keep these examples inside their newly created locations.


### Use one rule for storing and recovering text

A name on a roster may contain more than the English letters used in the first example. A **character encoding** is a rule for turning text into stored bytes and turning those bytes back into text. A **byte** is a small unit of stored data. **UTF-8** is the encoding used throughout these file examples.

```java
StandardCharsets.UTF_8
```

This expression is a **library constant**, a named value supplied for reuse. `StandardCharsets` is a library class and `UTF_8` is its constant; neither spelling is a Java keyword. The expression will be an argument to both the write and read operations. Using the same rule in both directions lets the later read recover accented and non-Latin text correctly.

Encoding concerns stored bytes. A Java String’s `length()` counts its text storage positions, called **UTF-16 code units**. For the plain English roster names, each letter and each newline occupies one such position. That count is not a general file byte count, and some visible symbols use more than one String position. Our exact-text check will compare the whole String instead of treating a length match as proof of identical text.


### Save the complete text, then recover it

A Path gives an operation a location. It does not supply the text to save. **Whole-file writing** takes a String and stores it as a small file’s complete contents. This illustrative line assumes the `file` path and handler from the complete example below:

```java
Files.writeString(file, "old message", StandardCharsets.UTF_8);
```

The first argument identifies the file. The second provides the text. The third specifies the encoding. With no extra write option, `writeString` creates a missing file or replaces an existing file’s contents. It does not add a newline to the supplied String.

A second ordinary write changes the entire contents:

```java
Files.writeString(file, "new", StandardCharsets.UTF_8);
```

The shorter text replaces `old message`; the old trailing letters do not remain. A normal write of `""`, the empty String, similarly leaves an empty file.

**Whole-file reading** recovers all the text from a small file into one String:

```java
String recovered = Files.readString(file, StandardCharsets.UTF_8);
```

`readString` uses the path to find the file and the encoding to decode its bytes. Its returned String includes the stored newlines, if any. The variable receives that text only after a successful read. A failed read throws rather than providing an invented empty result.

Writing and reading are different from printing. Printing sends text to the displayed output; it does not change the stored file. The complete replacement example below prints the returned read directly.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-write-");
    Path file = directory.resolve("note.txt");
    Files.writeString(file, "old message", StandardCharsets.UTF_8);
    Files.writeString(file, "new", StandardCharsets.UTF_8);
    System.out.println(Files.readString(file, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
new
```

Both writes use the same path. The second replaces the first text; the read returns only `new`. `println` adds the displayed line ending after that returned text. There is no stored newline in this particular file.

The complete program repeats the imports and new-directory setup, so it does not need a variable left by an earlier cell. These whole-file helpers open and close the resources they use internally. A very large file can be unsuitable for loading into one String; the next lesson teaches a line-by-line alternative.


### Connect the path to the created file

We can now combine location, creation, writing, and reading without leaving an unexplained operation. The following model first inspects a relative name, then creates its own directory and resolves the same fixed child name there. It prints the resolved name before writing. Finally, it writes Maya and a newline, then reads that saved text back.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path relative = Path.of("roster.txt");
    System.out.println("Relative name: " + relative.getFileName());
    System.out.println("Absolute: " + relative.isAbsolute());
    Path directory = Files.createTempDirectory("csc239-path-demo-");
    Path roster = directory.resolve("roster.txt");
    System.out.println("Resolved name: " + roster.getFileName());
    Files.writeString(roster, "Maya\n", StandardCharsets.UTF_8);
    System.out.print(Files.readString(roster, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Relative name: roster.txt
Absolute: false
Resolved name: roster.txt
Maya
```

The first two reports describe the relative Path. The third report comes from the resolved child’s name; it occurs before the write and does not prove creation. The final line is different evidence: it comes from reading the text after the write created the file. The animation keeps that order; its last state recalls the earlier name report while showing the current read.


<details class="animation-panel" open>
<summary>Path value and created entry — show or hide animation</summary>
<p><img src="media/01_paths_and_text_files/path_value_and_created_entry.gif" alt="Path.of creates a relative path value; it does not create a file. createTempDirectory creates the new practice directory. resolve creates a child path value inside that directory; the file is still absent. writeString creates roster.txt and stores Maya followed by a newline. The earlier name report was roster.txt; the current read now recovers Maya and its newline from that path." width="960" style="max-width:100%;height:auto;"></p>
</details>

Path values identify locations. Directory creation creates the parent entry; writing creates the file. The parent is generated, but the chosen child name and recovered text remain predictable. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View the still explanation of path value and created entry](media/01_paths_and_text_files/path_value_and_created_entry_still.png).


### Check every recovered character and newline

The next model saves a greeting with `olá` and `山` on separate lines. These are teaching data for an encoding check. `original` holds the full intended text, including a final newline after each line. `recovered` holds the later read.

```java
recovered.equals(original)
```

This comparison asks whether every position matches, including line endings. A normal write returning successfully tells us that the operation completed. Reading and comparing adds a check of the actual recovered text. No comparison can establish that a future operation will also succeed.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-encoding-demo-");
    Path file = directory.resolve("greeting.txt");
    String original = "olá\n山\n";
    Files.writeString(file, original, StandardCharsets.UTF_8);
    String recovered = Files.readString(file, StandardCharsets.UTF_8);
    System.out.print(recovered);
    System.out.println("Exact text: " + recovered.equals(original));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
olá
山
Exact text: true
```

The recovered text retains both the accented word and the non-Latin character. `Exact text: true` confirms equality with the complete original String, including both newlines. The diagram treats the file as encoded bytes without inventing a byte array or claiming that String length measures those bytes.


<details class="animation-panel" open>
<summary>Utf8 text round trip — show or hide animation</summary>
<p><img src="media/01_paths_and_text_files/utf8_text_round_trip.gif" alt="The original String contains olá and 山, each followed by a newline. UTF-8 encoding supplies bytes for the file write. The Path names the file; its bytes are the separately stored contents. Reading with UTF-8 decodes the same text. equals reports true for the complete recovered String, including both newlines." width="960" style="max-width:100%;height:auto;"></p>
</details>

Writing and reading use the same encoding. The recovered text matches every stored character position and newline in this example. The diagram describes bytes conceptually and does not claim that String length is a byte count. The sequence repeats every 12.6 seconds. Hide the panel to remove visible motion.

[View the still explanation of utf8 text round trip](media/01_paths_and_text_files/utf8_text_round_trip_still.png).


### Add text while keeping what is already saved

Replacing an entire roster would lose its earlier names when a new person arrives. **Append mode** writes at the end of the existing contents. In an illustrative write, the extra argument selects that behavior:

```java
Files.writeString(file, "kit\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
```

`StandardOpenOption.APPEND` is a library constant, not a keyword. Supplying this option alone expects the file to exist. That is why the complete example first creates the file with a normal write of `"map\n"`.

The two Strings supply their own line breaks. The newline after map separates it from kit; the newline after kit ends the final line. Append inserts no extra separator. Appending `""` adds no text, while a normal write of `""` replaces all earlier contents. The complete model below uses one initial write and one append.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-append-");
    Path file = directory.resolve("labels.txt");
    Files.writeString(file, "map\n", StandardCharsets.UTF_8);
    Files.writeString(file, "kit\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    System.out.print(Files.readString(file, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
map
kit
```

The saved String is `"map\nkit\n"`. `print` uses its existing newlines rather than adding a new one. The first write and append target the same file. This makes a useful distinction when deciding what a repeated operation should do.


### Distinguish another append from another complete setup

A repeated append against the same path adds the text again. A complete run of our example also repeats `createTempDirectory`, which creates a different directory before either write. That complete setup therefore starts a separate fixture.

The next model shows both behaviors in one complete program execution. It creates a first fixture, appends twice to that file, then creates a second fixture and performs only one append there. The labels “first fixture” and “fresh fixture” identify two locations inside this one run. They do not mean the program was executed twice.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
import java.nio.file.StandardOpenOption;
try {
    Path firstDirectory = Files.createTempDirectory("csc239-replay-demo-");
    Path firstFile = firstDirectory.resolve("labels.txt");
    Files.writeString(firstFile, "map\n", StandardCharsets.UTF_8);
    Files.writeString(firstFile, "kit\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    System.out.println("First fixture:");
    System.out.print(Files.readString(firstFile, StandardCharsets.UTF_8));
    Files.writeString(firstFile, "kit\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    System.out.println("Same file after another append:");
    System.out.print(Files.readString(firstFile, StandardCharsets.UTF_8));
    Path secondDirectory = Files.createTempDirectory("csc239-replay-demo-");
    Path secondFile = secondDirectory.resolve("labels.txt");
    Files.writeString(secondFile, "map\n", StandardCharsets.UTF_8);
    Files.writeString(secondFile, "kit\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    System.out.println("Fresh fixture:");
    System.out.print(Files.readString(secondFile, StandardCharsets.UTF_8));
    System.out.println("Earlier fixture is separate: " + Files.readString(firstFile, StandardCharsets.UTF_8).equals("map\nkit\nkit\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
First fixture:
map
kit
Same file after another append:
map
kit
kit
Fresh fixture:
map
kit
Earlier fixture is separate: true
```

The first file grows from two lines to three because its second append uses the same path. The second file starts from its own initial write and has two lines. The final `true` report confirms that the earlier file still has three lines. Creating another fixture does not erase or modify the first one. This is the distinction you will use when testing a whole-cell replay.


<details class="animation-panel" open>
<summary>Same file append vs fresh fixture — show or hide animation</summary>
<p><img src="media/01_paths_and_text_files/same_file_append_vs_fresh_fixture.gif" alt="The first fixture is created and contains map followed by kit. A second append to the same firstFile adds another kit line. Repeating the complete setup creates a separate second fixture. The second fixture starts with map and one kit; the first fixture retains its earlier three lines." width="960" style="max-width:100%;height:auto;"></p>
</details>

Two appends against the same path accumulate text. Repeating directory creation and starting writes builds a separate file. This one complete program shows both fixture lifetimes; it does not pretend the kernel itself ran twice. The sequence repeats every 10.2 seconds. Hide the panel to remove visible motion.

[View the still explanation of same file append vs fresh fixture](media/01_paths_and_text_files/same_file_append_vs_fresh_fixture_still.png).


## Worked Example

### Give the coordinator a saved roster

Return to the club coordinator’s task. The first two people are Maya and Luis. Nora arrives later. The required file is `roster.txt`, inside this run’s new directory, with one newline after each name. The program should read that file back, print the names, and report the recovered String length.

**Choose a new location.** The `directory` variable receives a newly created practice directory. `roster` receives the result of resolving the fixed child name there. Those variables identify locations, not the names to store.

```java
Path directory = Files.createTempDirectory("csc239-text-");
Path roster = directory.resolve("roster.txt");
```

**Save and extend the roster.** The first write stores Maya and Luis. The second adds Nora while retaining the earlier text. Both use the same encoding and the same file path.

```java
Files.writeString(roster, "Maya\nLuis\n", StandardCharsets.UTF_8);
Files.writeString(roster, "Nora\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
```

**Recover the saved record.** The read creates the String to inspect. `print` uses the stored newlines. The following `println` reports the length of that String; it does not count the generated directory name or measure a file’s byte size.

```java
String text = Files.readString(roster, StandardCharsets.UTF_8);
System.out.print(text);
System.out.println("Characters: " + text.length());
```

Each snippet is part of the complete model below, including its imports and failure handler. On success, the operations happen in that order. If an operation throws `IOException`, the remaining normal work in the `try` is skipped and the handler reports the problem.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Maya\nLuis\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Nora\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Maya
Luis
Nora
Characters: 15
```

The read-back contains Maya, Luis, and Nora with a newline after each name. Each name has four letters, so the three names plus three newline characters have String length 15. For these names, each letter occupies one String position. In general, String length counts UTF-16 code units; it does not measure stored file bytes.


<details class="animation-panel" open>
<summary>Replace append and newlines — show or hide animation</summary>
<p><img src="media/01_paths_and_text_files/replace_append_and_newlines.gif" alt="The initial write stores Maya and Luis with final newlines. APPEND preserves that text and adds Nora plus a newline. The read returns the complete three-line String. print uses the stored newline characters; length reports 15 for this String." width="960" style="max-width:100%;height:auto;"></p>
</details>

The first write stores two lines. APPEND adds the third without replacing them. Reading returns every newline, and print uses those stored characters. A normal write has different replacement behavior, taught in the earlier write/read section; this executed example contains no later replacement write. The sequence repeats every 10.2 seconds. Hide the panel to remove visible motion.

[View the still explanation of replace append and newlines](media/01_paths_and_text_files/replace_append_and_newlines_still.png).


## Guided Practice

Apply the same file rules to different inputs. Start with a prediction, then complete, change, and repair programs. The empty Java work cells are intentional; fill them when a task asks you to construct or repair a complete program.


### Predict the recovered roster

Before running the program, write every output line in order, including the number after `Characters:`. Write the exact file text after the first write and after the append, using `\n` to show each newline. Explain which statement creates the directory, which statement only builds the file path, and why the first write must happen before this append.


In [ ]:
Your response:

Predicted complete output:


File text after first write, with newlines shown:


File text after append, with newlines shown:


Directory creation versus path resolution:


Why the first write comes before append:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the program above and compare its output with your prediction without erasing the original. Then run that complete cell once more. Record both outputs and explain why the last stored newline places the count on a new line, and why the earlier run’s file gains no additional name.


In [ ]:
Your response:

Actual first-run output:


What matched or changed, and why:


Actual whole-cell replay output:


Final stored newline and count position:


Why the earlier file stays unchanged:


### Trace paths, text and failure handling


Trace the roster program in a table with one row for directory creation, path resolution, the first write, append, read and printing. Record what each row creates, what path or String value it produces, and whether it changes stored text. Show the two stored Strings with `\n`. Explain why all three text operations select `StandardCharsets.UTF_8`, and why `text.length()` measures the recovered String rather than the file’s stored byte count. Identify the `IOException` handler. If the first write fails with that exception, explain which later statements in the `try` are skipped and what the handler prints.


In [ ]:
Your response:

Operation/state table: creation, resolution, first write, append, read, printing:


Why all text operations use UTF-8:


What String length measures:


Statements skipped if the first write fails:


What the handler reports:


<details>
<summary>Show answer</summary>

The output contains Iris, Bo and Eli on separate lines, then `Characters: 12`. Each name plus its newline contributes 5, 3 and 4 characters, so 5 + 3 + 4 = 12. The final `\n` in the recovered String moves the count to the next line.

Each complete run calls `createTempDirectory` again, so it writes a new file in a distinct directory. The earlier file remains unchanged; temporary does not mean it disappears when the cell ends. Repeating an append against one already-created file would instead add more text to that file.

Creating the temporary directory creates a distinct place for this run. Resolving `roster.txt` creates its path value, not the file. The first write creates the file and stores `Iris\nBo\n`; the append changes it to `Iris\nBo\nEli\n`. `readString` reads all of this small file into `text`; printing does not write to the file. UTF-8 is the same text encoding for both writes and the read, so the read uses the rule that stored the text. `text.length()` counts the recovered String; it is not a measurement of stored bytes.

If the first write throws `IOException`, control skips the append, read and normal prints. The handler prints `File problem: ` followed by that exception’s message. Successful runs do not enter the handler. These whole-file helpers open and close the resources they use.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Iris
Bo
Eli
Characters: 12
```

Common error: Counting the letters but leaving out each stored newline. Treating `resolve` as a file-creation operation. Expecting append to replace the first two names.

</details>


### Interpret a path without creating its file

Read the complete program below before running it. Predict all three printed lines. Identify which path is relative and explain what `resolve` will combine. Distinguish the statement that creates a directory from statements that only describe a location.


In [ ]:
Your response:

Predicted three lines:


Relative path and resolved path reasoning:


Operation that creates something:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-path-");
    Path relative = Path.of("schedule.txt");
    Path file = directory.resolve("schedule.txt");
    System.out.println("Relative absolute: " + relative.isAbsolute());
    System.out.println("Resolved absolute: " + file.isAbsolute());
    System.out.println("Name: " + file.getFileName());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Compare the actual output with your prediction. Explain why a printed final name does not establish file creation and why the generated full directory name is unsuitable for a fixed expected result.


In [ ]:
Your response:

Actual three lines:


What was created and what was only represented:


Why printing the name does not prove file creation:


Why the full generated directory name is not fixed:


<details>
<summary>Show answer</summary>

`Path.of("schedule.txt")` is a relative path, so its first result is false. In this Workspace, the new temporary directory has an absolute path; resolving the fixed relative child `schedule.txt` gives an absolute path, so the second result is true. `getFileName()` returns the final name `schedule.txt`. Only `createTempDirectory` creates something on disk here: the new directory. Neither `Path.of` nor `resolve` creates the text file. A path can represent a location whose file does not yet exist. The generated directory name can change on every run, while the fixed final name remains suitable for comparison.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-path-");
    Path relative = Path.of("schedule.txt");
    Path file = directory.resolve("schedule.txt");
    System.out.println("Relative absolute: " + relative.isAbsolute());
    System.out.println("Resolved absolute: " + file.isAbsolute());
    System.out.println("Name: " + file.getFileName());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Relative absolute: false
Resolved absolute: true
Name: schedule.txt
```

Common error: Assuming that printing a path creates the file. Predicting one fixed generated temporary-directory name. Assuming the directory is removed when the cell ends.

</details>


### Complete a small task file

The draft below is incomplete and is shown for editing. Replace `CHILD_OPERATION`, `TEXT_ENCODING`, `WRITE_MODE` and `READ_OPERATION` using `resolve`, `UTF_8`, `APPEND` and `readString`, once each. Keep the supplied `Open\n` and `Close\n` Strings, imports, new directory and handler. Put the completed program in the empty work cell. Before running, predict the recovered lines and the `Matches:` result. 

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-task-");
    Path file = directory.CHILD_OPERATION("tasks.txt");
    Files.writeString(file, "Open\n", StandardCharsets.TEXT_ENCODING);
    Files.writeString(file, "Close\n", StandardCharsets.UTF_8, StandardOpenOption.WRITE_MODE);
    String text = Files.READ_OPERATION(file, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Matches: " + text.equals("Open\nClose\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Four replacements and their purposes:


Predicted recovered lines and Matches report:


Record the completed program’s output. Explain why the first line remains after the second write and why both newline characters belong in the equality check.


In [ ]:
Your response:

Actual output:


Why the first line remains:


Why both newline characters matter:


<details>
<summary>Show answer</summary>

The replacements are `CHILD_OPERATION` → `resolve`, `TEXT_ENCODING` → `UTF_8`, `WRITE_MODE` → `APPEND`, and `READ_OPERATION` → `readString`. The first write stores `Open\n`; the second appends `Close\n`. Reading returns `Open\nClose\n`, so the exact equality check is true. A different newline pattern would change the String even if the words looked similar. All required imports and the `IOException` handler are included.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-task-");
    Path file = directory.resolve("tasks.txt");
    Files.writeString(file, "Open\n", StandardCharsets.UTF_8);
    Files.writeString(file, "Close\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(file, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Matches: " + text.equals("Open\nClose\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Open
Close
Matches: true
```

Common error: Putting the reading operation where path resolution belongs. Replacing the second write’s append mode with default writing. Leaving out a newline when checking exact text.

</details>


### Append twice to the same file

The starter below performs one append. Add a second identical append of `Eli\n` immediately before `readString`, using the same `roster` path. Before running the complete modified cell, predict its lines and count and explain the change you will make.


In [ ]:
Your response:

Planned extra operation and location:


Predicted modified output:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Record the modified output and explain why the extra operation changes both the text and count.


In [ ]:
Your response:

Actual modified output:


How the second append changes the saved text and count:


Now plan a restoration test: remove the extra append, then run the restored complete program twice. Predict both outputs before editing or running it. Explain whether those two runs will share a directory.


In [ ]:
Your response:

Predicted first restored run:


Predicted second restored run:


Directory relationship and reason:


Return to the Java work cell above, remove only the added append, and run the restored complete program twice.


Record both restored outputs. Explain why two append calls within one complete run differ from two complete runs that each create a new directory.


In [ ]:
Your response:

Actual first restored output:


Actual second restored output:


Same-file repetition versus complete setup repetition:


<details>
<summary>Show answer</summary>

Within one run, both append calls target the same file. The second adds another four characters, `Eli\n`, so the recovered text is `Iris\nBo\nEli\nEli\n` and the count is 16. Restoring one append gives the complete program shown in the comparison case. Each restored whole-cell execution creates a new directory and prints the three names with count 12. It does not append to an earlier run’s file. Earlier directories remain after their cells finish.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Iris
Bo
Eli
Eli
Characters: 16
```

Common error: Adding the second append after reading, so `text` still holds the earlier contents. Changing the second append into a default write that replaces the names. Expecting the restored whole-cell replay to reuse the previous directory.

**Additional test: `Restored single append; run the complete program twice`.** Each complete run creates its own directory before writing. Both runs recover the same three lines with count 12; neither adds to the earlier file.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Iris
Bo
Eli
Characters: 12
```

</details>


### Repair a write that loses earlier names

The intended result is the Iris, Bo and Eli roster from your first practice program. Inspect this faulty draft before running it. Predict what remains after its second write and identify the operation that loses the earlier names. Plan a repair to that write’s mode only, then put the full repaired program in the empty Java cell.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


In [ ]:
Your response:

Predicted faulty output:


Operation that loses the earlier text:


Planned repair and predicted repaired output:


Run your repaired complete program. Record the output and explain how the repair preserves the earlier names.


In [ ]:
Your response:

Actual repaired output:


Why the earlier text is preserved:


Plan an additional test of the repaired program. Change only its first written String to `""`, but retain that first write. Predict the recovered text and count before editing and running. Explain why the first write still has a purpose.


In [ ]:
Your response:

Predicted empty-initial-file output:


Why retaining the first write matters:


Make that one input change in your repaired Java work cell above and run the complete program.


Record the additional test’s output. Explain why a shorter result in this case does not show that the original replacement fault returned.


In [ ]:
Your response:

Actual additional-test output:


Why shorter text is correct in this test:


<details>
<summary>Show answer</summary>

The faulty second write has no append option. It replaces the file with `Eli\n`, losing Iris and Bo; the count becomes 4. Add `StandardOpenOption.APPEND` to that call to recover all three names and count 12. In the additional test, the first write creates an empty file, so appending `Eli\n` correctly produces only Eli and count 4. That short output has a different reason from the faulty program: there were no earlier names to preserve. The first write must remain because this append option alone does not create a missing file.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "Iris\nBo\n", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Iris
Bo
Eli
Characters: 12
```

Common error: Changing only the printed count while leaving the lost text unfixed. Removing the first write instead of creating an empty file for the extra test. Treating identical short output as proof that two programs performed the same operations.

**Additional test: `Repaired append after an empty first write`.** The normal write of an empty String creates the file. The append then adds Eli and a newline. There is no lost earlier text in this case.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-text-");
    Path roster = directory.resolve("roster.txt");
    Files.writeString(roster, "", StandardCharsets.UTF_8);
    Files.writeString(roster, "Eli\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(roster, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Characters: " + text.length());
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Eli
Characters: 4
```

</details>


## Independent Practice

### Save and recover a menu

Write a complete program that creates a new temporary directory with prefix `csc239-menu-` and resolves `menu.txt` inside it. Write `café` and `tea` on separate newline-terminated lines using UTF-8. Append `水` followed by a newline using `APPEND`, then read the whole file using UTF-8. Print the recovered text, followed by `Read-back matches: ` and the result of comparing it with the exact expected three-line String. Include all required imports and an `IOException` handler that prints `File problem: ` followed by the message. Keep every write inside this program’s new directory. Before writing and running, plan the operations and predict the full String and output. The next step adds replacement and empty-text tests.


In [ ]:
Your response:

Plan: location, initial write, append, read, comparison, failure handler:


Predicted complete String with newlines shown:


Predicted output:


Record the recovered text and comparison result from your program. Explain how matching encodings support the accented and non-Latin text, and why exact String equality gives stronger evidence than only looking at the words.


In [ ]:
Your response:

Actual recovered text and comparison:


Encoding explanation:


What the complete equality check establishes:


### Test replacement and empty text

Test four separate complete versions of your menu program. Start each version by creating its own new directory. For the original version, keep the three-line menu. For replacement, add a normal UTF-8 write of `"juice\n"` after the existing append and before the read. For empty append, return to the original version and append `""` at that same point. For an empty file, return to the original version and add a normal UTF-8 write of `""` at that point. Predict the exact recovered String for all four versions and the final original-menu replay before the first run. Run the cases in order, comparing each result against its prediction with `equals`, then record the results by case. Record the actual printed text and boolean, including whether any menu lines appear. Explain why an empty append and an empty normal write differ. Finally rerun the original complete cell and explain its new-directory behavior using your earlier same-file double-append test.


In [ ]:
Your response:

Original menu:


Replacement with juice and a newline:


Empty append:


Empty file:


Whole-cell replay of the original menu:


Run all four complete versions and the final original-menu replay in the stated order. Use the Java work cell below for each complete version. Each version must create its own directory. Keep your recorded predictions unchanged so the later comparison remains meaningful.


After the final run, record the actual text, equality result, and reasoning for every named case. Explain the difference between empty append and empty replacement. Compare the original-menu replay with the earlier same-file double append.


In [ ]:
Your response:

Original menu:


Replacement with juice and a newline:


Empty append:


Empty file:


Whole-cell replay of the original menu:


Why empty append and empty replacement differ:


Whole-cell replay versus same-file double append:


<details>
<summary>Show answer</summary>

The normal first write creates `menu.txt` with `café\ntea\n`; append adds `水\n`. Matching UTF-8 on writing and reading preserves the text. Reading returns the complete String `café\ntea\n水\n`, including the newline after each menu item, so the equality check is true. The output first shows the recovered menu, then reports the comparison. The handler is available for an I/O failure and is not entered on this successful path. The complete example needs only the imports shown and its own new directory.

The original menu recovers `café\ntea\n水\n` and compares equal. A normal write of `juice\n` replaces the entire earlier menu, leaving only that line. Appending an empty String adds nothing, so the original three-line menu remains. A normal write of an empty String replaces all contents; reading succeeds with `""`, and `print(text)` prints no menu line. Each case checks its own predicted full String, so a true result means that case matches its stated expectation.

The full-program rerun begins with a different newly created directory and cannot add another line to the earlier file. Two append operations inside one run target the same file and do add twice.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-menu-");
    Path menu = directory.resolve("menu.txt");
    Files.writeString(menu, "café\ntea\n", StandardCharsets.UTF_8);
    Files.writeString(menu, "水\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(menu, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Read-back matches: " + text.equals("café\ntea\n水\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
café
tea
水
Read-back matches: true
```

Common error: Writing the expected String to the console without reading the actual file. Using different encodings for the write and read. Checking only the visible words while omitting newline differences. Appending before creating the file.

**Additional test: Replace the saved menu with juice and a newline.** The normal write replaces all old contents with `juice\n`. Compare against that replacement String, so the observed true result confirms this case rather than the original menu.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-menu-");
    Path menu = directory.resolve("menu.txt");
    Files.writeString(menu, "café\ntea\n", StandardCharsets.UTF_8);
    Files.writeString(menu, "水\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    Files.writeString(menu, "juice\n", StandardCharsets.UTF_8);
    String text = Files.readString(menu, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Read-back matches: " + text.equals("juice\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
juice
Read-back matches: true
```

**Additional test: Append an empty String to the original menu.** Appending `""` adds no character. The exact original three-line String, including its last newline, remains and compares equal.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-menu-");
    Path menu = directory.resolve("menu.txt");
    Files.writeString(menu, "café\ntea\n", StandardCharsets.UTF_8);
    Files.writeString(menu, "水\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    Files.writeString(menu, "", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    String text = Files.readString(menu, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Read-back matches: " + text.equals("café\ntea\n水\n"));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
café
tea
水
Read-back matches: true
```

**Additional test: Replace the original menu with an empty String.** The normal write of `""` leaves an empty file. Reading returns `""`; no menu text is printed, and the empty-String equality check reports true.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-menu-");
    Path menu = directory.resolve("menu.txt");
    Files.writeString(menu, "café\ntea\n", StandardCharsets.UTF_8);
    Files.writeString(menu, "水\n", StandardCharsets.UTF_8, StandardOpenOption.APPEND);
    Files.writeString(menu, "", StandardCharsets.UTF_8);
    String text = Files.readString(menu, StandardCharsets.UTF_8);
    System.out.print(text);
    System.out.println("Read-back matches: " + text.equals(""));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Read-back matches: true
```

</details>


## Summary

A Path describes a location; a file operation creates or changes what is stored there. Resolving a fixed child name gives this example a file path inside its own new directory. A later complete run creates another directory instead of silently reusing the earlier fixture.

Matching UTF-8 write and read operations use the same rule for stored text. A normal whole-file write creates or replaces contents; APPEND preserves existing text and adds the supplied String at its end. Neither operation invents a newline. Read-back and exact String equality check the saved result, including its line endings.

An IOException reports failure instead of a usable result. A matching handler can report that problem, while the unfinished normal operations are skipped.


Close the answer panels and use memory. Explain why `Path.of` does not create a file, why a normal write differs from append, and why repeating an append differs from repeating this example’s complete setup.


In [ ]:
Your response:

Path value versus creation:


Replacement versus append:


Same-file repetition versus complete setup:


<details>
<summary>Show answer</summary>

`Path.of` builds a value describing a location. File creation is a separate operation, such as a normal write to a missing file. A normal write replaces contents, whereas append adds at the existing end. Repeating append against the same file grows that file. Repeating our complete setup creates a separate directory and a new starting file. A common error is assuming that the word “temporary” means the earlier file disappeared.

</details>


## Reflection

A saved attendance list needs occasional full corrections and frequent new entries. Decide when replacement is appropriate and when append is appropriate. Describe an exact read-back test that includes accented and non-Latin text and newline boundaries. Explain what your comparison can establish about the saved result.


In [ ]:
Your response:

When to replace and when to append:


Exact input and expected read-back test:


What a successful comparison establishes:


A dictionary also stores meaningful text, but its entries are organized as lines. In the next lesson, you will keep this explicit path and encoding setup while using buffered readers and writers to process those lines and answer word-prefix questions.


## Supplemental Reading

- [Java 21 Path API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Path.html) explains location construction and resolution.
- [Java 21 Files API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) documents temporary directories and whole-file text operations.
- [Standard character encodings](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/charset/StandardCharsets.html) identifies UTF-8.
- [File open options](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/StandardOpenOption.html) explains APPEND and other modes.

- [Java String length and equality](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#length()) explains String positions; [equals](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#equals(java.lang.Object)) specifies exact text comparison.
- [IOException](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/IOException.html) identifies the checked type used for input/output failures.
